# Financial Classification: Cross-Variant Model Comparison

This notebook trains all four models (Logistic Regression, Random Forest, SVM, Neural Network) under four preprocessing variants and produces the comparison tables / figures for the final report.

| Variant | Data |
|---------|------|
| `d1` | Personal Finance Dataset only (Faker-text descriptions, clean labels) |
| `full` | D1 + D2 combined (proposal baseline) |
| `d2` | Personal Transactions UserID only (real text, augmentation-noised labels) |
| `d2_clean` | D2 with modal-category denoising (clean ceiling experiment) |

**Note**: Run this in Google Colab for best performance. Total training time is ~25 minutes for all 16 (variant × model) combinations.

## 1. Setup Environment

In [ ]:
!git clone https://github.com/darielgu/financial-classification.git
%cd financial-classification

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib
from sklearn.metrics import f1_score

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 2. Data Preprocessing

Generates `data/processed_<variant>/{train,val,test}.csv` for every variant.

In [ ]:
import subprocess, sys

VARIANTS = ["d1", "full", "d2", "d2_clean"]

for v in VARIANTS:
    print(f"\n=== Preprocessing variant: {v} ===")
    subprocess.run([sys.executable, "-m", "src.data_preprocessing", "--variant", v], check=True)

## 3. Train All Models on All Variants

Loops over (variant, model) combinations. Skips combos whose `joblib` artifact already exists. Set `FORCE_RETRAIN = True` to retrain everything.

In [ ]:
import os, subprocess, sys
from pathlib import Path

MODEL_MODULES = ["src.logistic_regression", "src.random_forest", "src.svm", "src.neural_network"]
FORCE_RETRAIN = False

def model_artifact(variant: str, module: str) -> Path:
    return Path(f"models/{variant}/{module.rsplit('.', 1)[-1]}.joblib")

for variant in VARIANTS:
    print(f"\n{'='*60}\nVariant: {variant}\n{'='*60}")
    env = {**os.environ, "VARIANT": variant}
    for module in MODEL_MODULES:
        path = model_artifact(variant, module)
        if path.exists() and not FORCE_RETRAIN:
            print(f"  skip {module}: {path} already exists")
            continue
        print(f"\n--- training {module} ---")
        subprocess.run([sys.executable, "-m", module], env=env, check=True)

## 4. Cross-Variant Evaluation

Loads all 16 trained models (4 variants × 4 models), runs each on its own variant's test split, and aggregates metrics for the report.

In [ ]:
from src.evaluate import compute_metrics
from src.model_data import load_processed_splits, prepare_features
from src.neural_network import LabelDecodingClassifier  # required for joblib.load on neural_network.joblib

MODELS = [
    ("Logistic Regression", "logistic_regression"),
    ("Random Forest",       "random_forest"),
    ("SVM",                 "svm"),
    ("Neural Network",      "neural_network"),
]

def evaluate_variant(variant: str):
    """Return (results_df, predictions_dict, y_test) for one variant."""
    train_df, val_df, test_df = load_processed_splits(Path(f"data/processed_{variant}"))
    _, _, transformed, _ = prepare_features(train_df, [val_df, test_df], dense=True)
    (_, _), (x_test, y_test) = transformed

    rows = []
    preds = {}
    for label, fname in MODELS:
        path = Path(f"models/{variant}/{fname}.joblib")
        if not path.exists():
            print(f"  [{variant}] skip {label}: {path} not found")
            continue
        m = joblib.load(path)
        if isinstance(m, dict):  # SVM stores {"model": ..., "feature_transformer": ...}
            m = m["model"]
        y_pred = m.predict(x_test)
        preds[label] = y_pred
        metrics = compute_metrics(y_test, y_pred, model_name=label)
        metrics["variant"] = variant
        metrics["test_size"] = len(y_test)
        metrics["n_classes"] = int(np.unique(y_test).size)
        rows.append(metrics)
    return pd.DataFrame(rows), preds, y_test

In [ ]:
all_results = []
all_preds = {}
all_y_test = {}
for v in VARIANTS:
    print(f"\n=== Loading variant: {v} ===")
    df, preds, y_test = evaluate_variant(v)
    all_results.append(df)
    all_preds[v] = preds
    all_y_test[v] = y_test

results_df = pd.concat(all_results, ignore_index=True)
results_df = results_df[["variant", "model", "test_size", "n_classes",
                         "accuracy", "precision_macro", "recall_macro", "f1_macro"]]

print("\n" + "=" * 80)
print("CROSS-VARIANT RESULTS")
print("=" * 80)
print(results_df.to_string(index=False))

### Macro-F1 pivot table

In [ ]:
pivot_f1 = results_df.pivot(index="model", columns="variant", values="f1_macro").reindex(
    columns=VARIANTS, index=[m[0] for m in MODELS]
)
pivot_acc = results_df.pivot(index="model", columns="variant", values="accuracy").reindex(
    columns=VARIANTS, index=[m[0] for m in MODELS]
)

print("Macro-F1 by (model, variant):")
print(pivot_f1.round(4).to_string())
print("\nAccuracy by (model, variant):")
print(pivot_acc.round(4).to_string())

## 5. Visualizations

In [ ]:
VARIANT_COLORS = {"d1": "#FF6B6B", "full": "#4ECDC4", "d2": "#45B7D1", "d2_clean": "#9B59B6"}
models_order = [m[0] for m in MODELS]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Cross-Variant Model Performance", fontsize=16, fontweight="bold")

metrics = ["accuracy", "precision_macro", "recall_macro", "f1_macro"]
titles = ["Accuracy", "Precision (Macro)", "Recall (Macro)", "F1-Score (Macro)"]
x = np.arange(len(models_order))
width = 0.2

for ax, metric, title in zip(axes.flat, metrics, titles):
    for i, v in enumerate(VARIANTS):
        sub = results_df[results_df["variant"] == v].set_index("model").reindex(models_order)
        vals = sub[metric].values
        ax.bar(x + i * width, vals, width, label=v, color=VARIANT_COLORS[v], edgecolor="black", linewidth=0.8)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_ylabel("Score", fontsize=11)
    ax.set_xticks(x + width * 1.5)
    ax.set_xticklabels(models_order, rotation=15)
    ax.set_ylim(0, 1.05)
    ax.grid(axis="y", alpha=0.3)
    ax.legend(title="Variant", fontsize=9)

plt.tight_layout()
plt.savefig("models/cross_variant_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved figure to models/cross_variant_comparison.png")

In [ ]:
# Heatmap of macro-F1 (model x variant)
fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(pivot_f1, annot=True, fmt=".3f", cmap="YlGnBu", vmin=0, vmax=1, ax=ax, cbar_kws={"label": "Macro-F1"})
ax.set_title("Macro-F1 across Variants and Models", fontsize=14, fontweight="bold")
ax.set_xlabel("Variant")
ax.set_ylabel("Model")
plt.tight_layout()
plt.savefig("models/cross_variant_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved heatmap to models/cross_variant_heatmap.png")

## 6. Per-Class Detail (primary variant)

Detailed classification reports for the primary variant chosen for the final report. Adjust `PRIMARY_VARIANT` to switch.

In [ ]:
from sklearn.metrics import classification_report

PRIMARY_VARIANT = "full"
y_test_primary = all_y_test[PRIMARY_VARIANT]
preds_primary = all_preds[PRIMARY_VARIANT]

print("=" * 70)
print(f"DETAILED CLASSIFICATION REPORTS -- variant: {PRIMARY_VARIANT}")
print("=" * 70)
for label, _ in MODELS:
    if label not in preds_primary:
        continue
    print(f"\n{label}:")
    print("-" * 70)
    print(classification_report(y_test_primary, preds_primary[label], zero_division=0))

## 7. Summary and Key Findings

In [ ]:
print("=" * 70)
print("BEST (model, variant) BY MACRO-F1")
print("=" * 70)
best = results_df.sort_values("f1_macro", ascending=False).head(5)
print(best[["variant", "model", "accuracy", "f1_macro"]].to_string(index=False))

print("\n" + "=" * 70)
print("AVERAGE MACRO-F1 PER VARIANT (across 4 models)")
print("=" * 70)
avg_per_variant = results_df.groupby("variant")["f1_macro"].mean().reindex(VARIANTS)
print(avg_per_variant.round(4).to_string())

print("\n" + "=" * 70)
print("AVERAGE MACRO-F1 PER MODEL (across 4 variants)")
print("=" * 70)
avg_per_model = results_df.groupby("model")["f1_macro"].mean().reindex([m[0] for m in MODELS])
print(avg_per_model.round(4).to_string())

In [ ]:
print("KEY INSIGHTS")
print("-" * 70)
v1_avg = avg_per_variant.get("full", float("nan"))
v0_avg = avg_per_variant.get("d1", float("nan"))
v2_avg = avg_per_variant.get("d2", float("nan"))
v3_avg = avg_per_variant.get("d2_clean", float("nan"))

print(f"V0 (D1, Faker text)         avg macro-F1: {v0_avg:.4f}")
print(f"V1 (D1+D2, baseline)        avg macro-F1: {v1_avg:.4f}")
print(f"V2 (D2 alone, noisy labels) avg macro-F1: {v2_avg:.4f}")
print(f"V3 (D2 denoised)            avg macro-F1: {v3_avg:.4f}")

print(f"\nV3 vs V2 lift               : +{v3_avg - v2_avg:.4f}")
print(f"V2 vs V1 (delta from drop D1): {v2_avg - v1_avg:+.4f}")

print("\nModel characteristics:")
print("  - LR : Linear, interpretable, tuned on C")
print("  - RF : Non-linear ensemble, captures feature interactions")
print("  - SVM: Linear margin in TF-IDF space")
print("  - NN : MLP with capped oversampling, learns non-linear feature combos")

## 8. Export Results

In [ ]:
results_df.to_csv("models/cross_variant_results.csv", index=False)
pivot_f1.to_csv("models/cross_variant_f1_pivot.csv")
pivot_acc.to_csv("models/cross_variant_accuracy_pivot.csv")

print("Saved:")
print("  - models/cross_variant_results.csv     (long form)")
print("  - models/cross_variant_f1_pivot.csv    (model x variant)")
print("  - models/cross_variant_accuracy_pivot.csv")
print("\nLong-form results:")
print(results_df.to_string(index=False))